# Environment Setup and Validation

**Purpose**: Validate Python environment, GPU configuration, and create necessary directories

**Requirements**:
- Python 3.11 or 3.12
- NVIDIA GPU with CUDA 12.x
- 32 GB RAM
- 100+ GB free disk space

**Run this notebook FIRST before any data download or processing**

In [1]:
# AUTOMATIC INSTALLATION OF DEPENDENCIES
# This ensures all required packages are present
print("Installing dependencies... This may take a few minutes.")
%pip install -r ../requirements.txt

Installing dependencies... This may take a few minutes.
Note: you may need to restart the kernel to use updated packages.


## 1. Import Core Libraries and Check Versions

In [2]:
import sys
import os
import platform
from pathlib import Path

# Display system information
print("=" * 80)
print("SYSTEM INFORMATION")
print("=" * 80)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Architecture: {platform.machine()}")
print(f"Processor: {platform.processor()}")
print("=" * 80)

SYSTEM INFORMATION
Python version: 3.12.8 (main, Feb 16 2026, 19:49:14) [GCC 15.2.1 20260209]
Platform: Linux-6.18.9-arch1-2-x86_64-with-glibc2.43
Architecture: x86_64
Processor: 


## 2. Verify Required Packages

In [ ]:
# Dictionary of required packages with minimum versions
required_packages = {
    'numpy': '1.26.0',
    'pandas': '2.0.0',
    'geopandas': '0.14.0',
    'rasterio': '1.3.0',
    'shapely': '2.0.0',
    'pyproj': '3.4.0',
    'xgboost': '2.0.0',
    'lightgbm': '4.0.0',
    'sklearn': '1.3.0',
    'matplotlib': '3.7.0',
    'seaborn': '0.12.0',
    'shap': '0.43.0',
    'pystac_client': '0.7.0',
    'planetary_computer': '1.0.0'
}

# Optional packages (GPU-related)
optional_packages = {
    'cupy': '13.0.0',
}

print("\n" + "=" * 80)
print("PACKAGE VERIFICATION")
print("=" * 80)

missing_packages = []
version_mismatches = []

# Check required packages
for package, min_version in required_packages.items():
    try:
        if package == 'sklearn':
            import sklearn
            installed_version = sklearn.__version__
        elif package == 'pystac_client':
            import pystac_client
            installed_version = pystac_client.__version__
        elif package == 'planetary_computer':
            import planetary_computer
            installed_version = planetary_computer.__version__
        else:
            module = __import__(package)
            installed_version = module.__version__
        
        print(f"[OK] {package:20s} {installed_version:15s} (required: >={min_version})")
        
    except ImportError:
        print(f"[MISSING] {package:20s} NOT INSTALLED")
        missing_packages.append(package)
    except AttributeError:
        print(f"[WARNING] {package:20s} installed but version unknown")

# Check optional packages
print("\nOptional Packages (GPU):")
for package, min_version in optional_packages.items():
    try:
        if package == 'cupy':
            import cupy as cp
            installed_version = cp.__version__
            print(f"[OK] {package:20s} {installed_version:15s} (GPU support available)")
        else:
            module = __import__(package)
            installed_version = module.__version__
            print(f"[OK] {package:20s} {installed_version:15s} (optional)")
    except ImportError:
        print(f"[SKIP] {package:20s} NOT INSTALLED (will use CPU only)")

print("=" * 80)

if missing_packages:
    print(f"\n[ERROR] Missing required packages: {', '.join(missing_packages)}")
    print("Run: pip install -r ../requirements.txt")
else:
    print("\n[SUCCESS] All required packages are installed")

## 3. GPU Detection and Configuration

In [ ]:
import os

# Configurar variables de entorno para CUDA
os.environ['CUDA_PATH'] = '/opt/cuda'
os.environ['LD_LIBRARY_PATH'] = f"{os.environ['HOME']}/.local/lib:/opt/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

print("\n" + "=" * 80)
print("GPU CONFIGURATION")
print("=" * 80)

try:
    import cupy as cp
    
    # Get GPU device
    device = cp.cuda.Device(0)
    
    # Get GPU properties
    total_memory = device.mem_info[1] / (1024**3)  # Convert to GB
    free_memory = device.mem_info[0] / (1024**3)
    used_memory = total_memory - free_memory
    
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"].decode("utf-8")
    print(f"GPU Device: {gpu_name}")
    print(f"Total Memory: {total_memory:.2f} GB")
    print(f"Free Memory: {free_memory:.2f} GB")
    print(f"Used Memory: {used_memory:.2f} GB")
    
    # Test GPU computation
    print("\n[TEST] Running simple GPU computation...")
    a_gpu = cp.array([1, 2, 3])
    b_gpu = cp.array([4, 5, 6])
    c_gpu = a_gpu + b_gpu
    print(f"GPU Test Result: {c_gpu.get()}")
    print("[SUCCESS] GPU is functional")
    
    # Configure memory pool (use 75% of available memory)
    mempool = cp.get_default_memory_pool()
    max_memory_gb = total_memory * 0.75
    mempool.set_limit(size=int(max_memory_gb * 1024**3))
    print(f"\n[CONFIG] GPU memory limit set to {max_memory_gb:.2f} GB (75% of total)")
    
    gpu_available = True
    
except ImportError:
    print("[WARNING] CuPy not installed - GPU acceleration not available")
    print("[INFO] Processing will use CPU only (slower)")
    gpu_available = False
except Exception as e:
    print(f"[WARNING] GPU not available: {str(e)}")
    print("[INFO] Processing will use CPU only (slower)")
    gpu_available = False

print("=" * 80)

## 4. Configure Environment Variables

In [5]:
os.environ['OMP_NUM_THREADS'] = '8'
os.environ['OPENBLAS_NUM_THREADS'] = '8'
os.environ['MKL_NUM_THREADS'] = '8'
os.environ['NUMEXPR_NUM_THREADS'] = '8'

print("\n" + "=" * 80)
print("ENVIRONMENT CONFIGURATION")
print("=" * 80)
print(f"OMP_NUM_THREADS: {os.environ.get('OMP_NUM_THREADS')}")
print(f"OPENBLAS_NUM_THREADS: {os.environ.get('OPENBLAS_NUM_THREADS')}")
print(f"MKL_NUM_THREADS: {os.environ.get('MKL_NUM_THREADS')}")
print(f"NUMEXPR_NUM_THREADS: {os.environ.get('NUMEXPR_NUM_THREADS')}")
print("=" * 80)


ENVIRONMENT CONFIGURATION
OMP_NUM_THREADS: 8
OPENBLAS_NUM_THREADS: 8
MKL_NUM_THREADS: 8
NUMEXPR_NUM_THREADS: 8


## 5. Create Project Directory Structure

In [6]:
# Get project root directory
project_root = Path.cwd().parent

# Define directory structure
directories = {
    'data': project_root / 'data',
    'data_raw': project_root / 'data' / 'raw',
    'data_raw_sentinel2': project_root / 'data' / 'raw' / 'sentinel2',
    'data_raw_dem': project_root / 'data' / 'raw' / 'dem',
    'data_raw_boundaries': project_root / 'data' / 'raw' / 'boundaries',
    'data_interim': project_root / 'data' / 'interim',
    'data_processed': project_root / 'data' / 'processed',
    'models': project_root / 'models',
    'figures': project_root / 'figures',
    'outputs': project_root / 'outputs',
    'outputs_logs': project_root / 'outputs' / 'logs',
}

print("\n" + "=" * 80)
print("DIRECTORY STRUCTURE")
print("=" * 80)
print(f"Project Root: {project_root}")
print("\nCreating directories...")

for name, path in directories.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"[CREATED] {name:25s} -> {path}")

print("\n[SUCCESS] Directory structure created")
print("=" * 80)


DIRECTORY STRUCTURE
Project Root: /home/andre/Documents/GLOF_Andes_Project-Paper

Creating directories...
[CREATED] data                      -> /home/andre/Documents/GLOF_Andes_Project-Paper/data
[CREATED] data_raw                  -> /home/andre/Documents/GLOF_Andes_Project-Paper/data/raw
[CREATED] data_raw_sentinel2        -> /home/andre/Documents/GLOF_Andes_Project-Paper/data/raw/sentinel2
[CREATED] data_raw_dem              -> /home/andre/Documents/GLOF_Andes_Project-Paper/data/raw/dem
[CREATED] data_raw_boundaries       -> /home/andre/Documents/GLOF_Andes_Project-Paper/data/raw/boundaries
[CREATED] data_interim              -> /home/andre/Documents/GLOF_Andes_Project-Paper/data/interim
[CREATED] data_processed            -> /home/andre/Documents/GLOF_Andes_Project-Paper/data/processed
[CREATED] models                    -> /home/andre/Documents/GLOF_Andes_Project-Paper/models
[CREATED] figures                   -> /home/andre/Documents/GLOF_Andes_Project-Paper/figures
[CREATED] 

## 6. Check Disk Space

In [7]:
import shutil

print("\n" + "=" * 80)
print("DISK SPACE")
print("=" * 80)

# Get disk usage for project directory
total, used, free = shutil.disk_usage(project_root)

total_gb = total / (1024**3)
used_gb = used / (1024**3)
free_gb = free / (1024**3)
used_percent = (used / total) * 100

print(f"Drive: {project_root.drive if hasattr(project_root, 'drive') else 'N/A'}")
print(f"Total Space: {total_gb:.2f} GB")
print(f"Used Space: {used_gb:.2f} GB ({used_percent:.1f}%)")
print(f"Free Space: {free_gb:.2f} GB")

# Check if we have enough space (need at least 100 GB)
required_gb = 100
if free_gb < required_gb:
    print(f"\n[WARNING] Low disk space! Required: {required_gb} GB, Available: {free_gb:.2f} GB")
    print("Consider freeing up space before proceeding")
else:
    print(f"\n[OK] Sufficient disk space available ({free_gb:.2f} GB > {required_gb} GB)")

print("=" * 80)


DISK SPACE
Drive: 
Total Space: 277.49 GB
Used Space: 82.37 GB (29.7%)
Free Space: 180.95 GB

[OK] Sufficient disk space available (180.95 GB > 100 GB)


## 7. Load Study Area Configuration

In [8]:
# Add parent directory to path to import config
sys.path.insert(0, str(project_root))

from config_expanded_study_areas import EXPANDED_STUDY_AREAS, OUTCOMES

print("\n" + "=" * 80)
print("STUDY AREAS CONFIGURATION")
print("=" * 80)
completed = sum(1 for area in EXPANDED_STUDY_AREAS.values() if area['status'] == 'COMPLETED')
pending = sum(1 for area in EXPANDED_STUDY_AREAS.values() if area['status'] == 'PENDING')
optional = sum(1 for area in EXPANDED_STUDY_AREAS.values() if area['status'] == 'OPTIONAL')

print(f"Total Study Areas: {len(EXPANDED_STUDY_AREAS)}")
print(f"  - Completed: {completed}")
print(f"  - Pending: {pending}")
print(f"  - Optional: {optional}")
print("\nPending Areas by Priority:")
print("-" * 80)
for priority in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
    areas = [(name, cfg) for name, cfg in EXPANDED_STUDY_AREAS.items() 
             if cfg['status'] == 'PENDING' and cfg['priority'] == priority]
    if areas:
        print(f"\n{priority}:")
        for name, cfg in areas:
            print(f"  - {name:30s} | {cfg['description']:50s} | Lakes: {cfg['lakes_estimated']}")

print("\n" + "=" * 80)


STUDY AREAS CONFIGURATION
Total Study Areas: 12
  - Completed: 2
  - Pending: 8
  - Optional: 2

Pending Areas by Priority:
--------------------------------------------------------------------------------

CRITICAL:
  - cordillera_huayhuash           | Cordillera Huayhuash, Ancash-Lima border           | Lakes: 80

HIGH:
  - cordillera_raura               | Cordillera Raura, Huánuco-Lima border              | Lakes: 35
  - bolivia_cordillera_real        | Cordillera Real, La Paz Department, Bolivia        | Lakes: 300
  - chile_andes_centrales          | Andes Centrales, Región Metropolitana, Chile       | Lakes: 100

MEDIUM:
  - cordillera_central             | Cordillera Central, Junín Region                   | Lakes: 45
  - cordillera_huanzo              | Cordillera Huanzo, Arequipa-Ayacucho Region        | Lakes: 25
  - cordillera_urubamba            | Cordillera Urubamba, Cusco Region                  | Lakes: 20
  - ecuador_antisana               | Antisana Volcano, Napo-Pichi

## 8. Test Data Download Connection (Microsoft Planetary Computer)

In [9]:
import pystac_client
import planetary_computer

print("\n" + "=" * 80)
print("PLANETARY COMPUTER CONNECTION TEST")
print("=" * 80)

try:
    # Connect to Microsoft Planetary Computer STAC API
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )
    
    print(f"[OK] Connected to Planetary Computer")
    print(f"Catalog ID: {catalog.id}")
    print(f"Catalog Title: {catalog.title}")
    
    # List available collections
    print("\nAvailable Collections (sample):")
    for i, collection in enumerate(catalog.get_collections()):
        if i < 5:  # Show first 5
            print(f"  - {collection.id:30s} | {collection.title}")
        elif i == 5:
            print(f"  ... and {sum(1 for _ in catalog.get_collections()) - 5} more")
            break
    
    print("\n[SUCCESS] Planetary Computer connection is functional")
    pc_connection = True
    
except Exception as e:
    print(f"[ERROR] Could not connect to Planetary Computer: {str(e)}")
    print("[WARNING] Data download will not work without internet connection")
    pc_connection = False

print("=" * 80)


PLANETARY COMPUTER CONNECTION TEST
[OK] Connected to Planetary Computer
Catalog ID: microsoft-pc
Catalog Title: Microsoft Planetary Computer STAC API

Available Collections (sample):
  - daymet-annual-pr               | Daymet Annual Puerto Rico
  - daymet-daily-hi                | Daymet Daily Hawaii
  - 3dep-seamless                  | USGS 3DEP Seamless DEMs
  - 3dep-lidar-dsm                 | USGS 3DEP Lidar Digital Surface Model
  - fia                            | Forest Inventory and Analysis
  ... and 129 more

[SUCCESS] Planetary Computer connection is functional


## 9. Save Environment Report

In [ ]:
from datetime import datetime
import json

# Create environment report
report = {
    'timestamp': datetime.now().isoformat(),
    'python_version': sys.version,
    'platform': platform.platform(),
    'gpu_available': gpu_available,
    'planetary_computer_connection': pc_connection,
    'disk_space_gb': {
        'total': round(total_gb, 2),
        'used': round(used_gb, 2),
        'free': round(free_gb, 2)
    },
    'study_areas': {
        'total': len(EXPANDED_STUDY_AREAS),
        'completed': completed,
        'pending': pending,
        'optional': optional
    }
}

# Add GPU info only if available and variables exist
if gpu_available:
    try:
        import cupy as cp
        device = cp.cuda.Device(0)
        total_mem = device.mem_info[1] / (1024**3)
        max_mem = total_mem * 0.75
        
        gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode('utf-8')
        report['gpu'] = {
            'name': gpu_name,
            'compute_capability': f"{device.compute_capability[0]}.{device.compute_capability[1]}",
            'total_memory_gb': round(total_mem, 2),
            'memory_limit_gb': round(max_mem, 2)
        }
    except:
        pass

# Save report
report_path = directories['outputs_logs'] / f"environment_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 80)
print("ENVIRONMENT REPORT SAVED")
print("=" * 80)
print(f"Report saved to: {report_path}")
print("\nReport Summary:")
print(json.dumps(report, indent=2))
print("=" * 80)

## 10. Final Checklist

In [11]:
print("\n" + "=" * 80)
print("SETUP VALIDATION CHECKLIST")
print("=" * 80)

checks = [
    ("Python 3.11+", sys.version_info >= (3, 11)),
    ("All required packages installed", len(missing_packages) == 0),
    ("Sufficient disk space", free_gb >= required_gb),
    ("Directory structure created", all(p.exists() for p in directories.values())),
]

# Optional checks (don't fail if these don't pass)
optional_checks = [
    ("GPU available (optional)", gpu_available),
    ("Planetary Computer connection (optional)", pc_connection),
]

all_passed = True
for check_name, passed in checks:
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {check_name}")
    if not passed:
        all_passed = False

print("\nOptional Features:")
for check_name, passed in optional_checks:
    status = "[OK]" if passed else "[SKIP]"
    print(f"{status} {check_name}")

print("=" * 80)

if all_passed:
    print("\n" + "=" * 80)
    print("ENVIRONMENT SETUP COMPLETE - READY TO PROCEED")
    print("=" * 80)
    if gpu_available:
        print("\n[INFO] GPU acceleration is enabled")
    else:
        print("\n[INFO] GPU not available - will use CPU (slower but functional)")
    
    if pc_connection:
        print("[INFO] Planetary Computer connection working")
    else:
        print("[WARNING] Planetary Computer connection failed - check internet")
    
    print("\nNext step: Run 01_download_peru_huayhuash.ipynb (PILOT)")
else:
    print("\n" + "!" * 80)
    print("SETUP INCOMPLETE - FIX ERRORS BEFORE PROCEEDING")
    print("!" * 80)
    print("\nReview failed checks above and resolve issues")


SETUP VALIDATION CHECKLIST
[PASS] Python 3.11+
[PASS] All required packages installed
[PASS] Sufficient disk space
[PASS] Directory structure created

Optional Features:
[OK] GPU available (optional)
[OK] Planetary Computer connection (optional)

ENVIRONMENT SETUP COMPLETE - READY TO PROCEED

[INFO] GPU acceleration is enabled
[INFO] Planetary Computer connection working

Next step: Run 01_download_peru_huayhuash.ipynb (PILOT)
